In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q huggingface_hub psutil
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [4]:
!mkdir -p models
!hf download Qwen/Qwen2.5-3B-Instruct-GGUF qwen2.5-3b-instruct-fp16.gguf --local-dir models
!hf download Qwen/Qwen2.5-3B-Instruct-GGUF qwen2.5-3b-instruct-q4_k_m.gguf --local-dir models

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Error: File not found in repository.
URL: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-fp16.gguf
Hint: set HF_DEBUG=1 as environment variable for full traceback.

qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: ▍       |  131MB, 9.40MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: reconstructing file:   6%| |  130MB / 2.10GB   
qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: █▌      |  399MB, 28.4MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: ██      |  530MB, 41.6MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: ██▎     |  611MB, 48.6MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: ██▋     |  704MB, 55.7MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes: ███     |  801MB, 63.9MB/s  
qwen2.5-3b-instruct-q4_k_m.gguf: reconstructing file:  32%|▎|  668MB / 2.10GB, 4
qwen2.5-3b-instruct-q4_k_m.gguf: 

In [5]:
!hf repos ls-files Qwen/Qwen2.5-3B-Instruct-GGUF

Usage: hf repos [OPTIONS] [COMMAND] [ARGS]...
Try 'hf repos --help' for help.

Error: No such command 'ls-files'. Did you mean 'delete-files'?

Available commands for 'hf repos':
  branch                         Manage branches for a repo on the Hub.
  cp                             Copy files between local paths, repositories, and buckets.
  create                         Create a new repo on the Hub.
  delete                         Delete a repo from the Hub.
  delete-files                   Delete files from a repo on the Hub.
  duplicate                      Duplicate a repo on the Hub (model, dataset, or Space).
  list                           List all repos (models, datasets, spaces, buckets) with storage info.
  move                           Move a repository from a namespace to another namespace.
  settings                       Update the settings of a repository.
  tag                            Manage tags for a repo on the Hub.

Run 'hf repos --help' for full details. Di

In [6]:
from huggingface_hub import HfApi

api = HfApi()
files = [f for f in api.list_repo_files("Qwen/Qwen2.5-3B-Instruct-GGUF") if f.endswith(".gguf")]
print("\n".join(files))

qwen2.5-3b-instruct-fp16-00001-of-00002.gguf
qwen2.5-3b-instruct-fp16-00002-of-00002.gguf
qwen2.5-3b-instruct-q2_k.gguf
qwen2.5-3b-instruct-q3_k_m.gguf
qwen2.5-3b-instruct-q4_0.gguf
qwen2.5-3b-instruct-q4_k_m.gguf
qwen2.5-3b-instruct-q5_0.gguf
qwen2.5-3b-instruct-q5_k_m.gguf
qwen2.5-3b-instruct-q6_k.gguf
qwen2.5-3b-instruct-q8_0.gguf


In [7]:
!hf download Qwen/Qwen2.5-3B-Instruct-GGUF qwen2.5-3b-instruct-q8_0.gguf --local-dir models


qwen2.5-3b-instruct-q8_0.gguf: downloading bytes: ██████▌   | 2.39GB,  178MB/s  
qwen2.5-3b-instruct-q8_0.gguf: downloading bytes: ████████▎ | 3.01GB,  199MB/s  
qwen2.5-3b-instruct-q8_0.gguf: downloading bytes: ████████▋ | 3.13GB,  200MB/s  
qwen2.5-3b-instruct-q8_0.gguf: downloading bytes: █████████▋| 3.51GB,  145MB/s  
qwen2.5-3b-instruct-q8_0.gguf: downloading bytes: ██████████| 3.52GB,  127MB/s  
qwen2.5-3b-instruct-q8_0.gguf: reconstructing file: 100%|█| 3.62GB / 3.62GB,  18
✓ Downloaded
  path: /kaggle/working/models/qwen2.5-3b-instruct-q8_0.gguf


In [3]:
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PROMPTS_10 = [
    "Write a Python function to detect a cycle in a directed graph using DFS.",
    "Explain why TCP uses a three-way handshake instead of two in 3 sentences.",
    "Convert this to JSON: User Alice (ID 42) ordered 3 keyboards on Nov 12.",
    "Solve step by step: A train travels 120 km at 60 km/h and returns at 40 km/h. Average speed?",
    "List 4 differences between B-Trees and LSM-Trees for write-heavy databases.",
    "Write a SQL query to find the second highest salary per department.",
    "Summarize the core trade-off of CAP theorem for distributed systems.",
    "Debug this snippet: `def append_item(x, lst=[]): lst.append(x); return lst`",
    "Translate into formal French: 'Please confirm receipt of the attached invoice.'",
    "Give 3 edge cases to test a rate-limiter middleware.",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def run_bench(label, quant_4bit=False):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    vram_before = torch.cuda.memory_allocated() / (1024 ** 2)

    if quant_4bit:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

    model.eval()
    vram_loaded = torch.cuda.memory_allocated() / (1024 ** 2)

    total_new_tokens = 0
    total_time = 0.0
    outs = []

    for i, p in enumerate(PROMPTS_10):
        inputs = tokenizer(p, return_tensors="pt").to("cuda")
        in_len = inputs["input_ids"].shape[1]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=150, do_sample=False)
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        new_toks = gen.shape[1] - in_len
        total_new_tokens += new_toks
        total_time += dt
        outs.append({"id": i, "tokens": new_toks, "text": tokenizer.decode(gen[0][in_len:], skip_special_tokens=True).strip()})

    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(json.dumps({
        "setting": label,
        "model_vram_mb": round(vram_loaded - vram_before, 1),
        "peak_vram_mb": round(peak_vram, 1),
        "avg_tok_per_sec": round(total_new_tokens / total_time, 2),
        "total_tokens": total_new_tokens,
    }, indent=2))

    with open(f"outputs_{label}.json", "w") as f:
        json.dump(outs, f, indent=2)

    del model
    torch.cuda.empty_cache()

run_bench("FP16", quant_4bit=False)
run_bench("INT4_NF4", quant_4bit=True)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{
  "setting": "FP16",
  "model_vram_mb": 3533.8,
  "peak_vram_mb": 3552.2,
  "avg_tok_per_sec": 14.61,
  "total_tokens": 1432
}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

{
  "setting": "INT4_NF4",
  "model_vram_mb": -1938.8,
  "peak_vram_mb": 3542.9,
  "avg_tok_per_sec": 10.6,
  "total_tokens": 1438
}


In [2]:
!pip install -q -U bitsandbytes>=0.46.1

In [9]:
!pip uninstall -y llama-cpp-python
!CMAKE_ARGS="-DGGML_CUDA=on" pip install --no-cache-dir llama-cpp-python

Found existing installation: llama_cpp_python 0.3.37
Uninstalling llama_cpp_python-0.3.37:
  Successfully uninstalled llama_cpp_python-0.3.37
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 84.4 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> llama-cpp-python
